# Task 2 — Customer Segmentation Analysis

## Objective
Group customers with similar purchasing characteristics and turn the groups into practical business segments.

### Important dataset note
This retail dataset contains **1,000 transactions and 1,000 unique Customer IDs**, so each customer appears once. Because of that, classic RFM **Frequency** and **Recency** features would be constant or nearly uninformative for this particular dataset. To avoid creating misleading results, this project segments customers using the useful fields that are actually available: **Age, Quantity, Price per Unit, and Total Amount**.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn and Scikit-learn.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv("../data/retail_sales_dataset.csv")
df["Date"] = pd.to_datetime(df["Date"])

print("Shape:", df.shape)
print("Unique customers:", df["Customer ID"].nunique())
display(df.head())
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())


## 1. Understand the available customer information

The dataset provides customer demographics and transaction behaviour. Since each Customer ID occurs once, the analysis treats each row as one customer profile.

The segmentation variables are:
- **Age** — demographic characteristic
- **Quantity** — number of units purchased
- **Price per Unit** — price level of the purchase
- **Total Amount** — transaction value


In [ ]:
features = ["Age", "Quantity", "Price per Unit", "Total Amount"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), features):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"{col} Distribution")
plt.tight_layout()
plt.show()


**Observation:** The four variables have different scales and distributions. Standardization is therefore needed before K-Means so that a large-number variable such as Total Amount does not dominate the distance calculation.


In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(df[features])

silhouette_scores = {}
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X)
    silhouette_scores[k] = silhouette_score(X, labels)

plt.figure(figsize=(8, 5))
plt.plot(list(silhouette_scores.keys()), list(silhouette_scores.values()), marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.title("Choosing the Number of Clusters")
plt.grid(alpha=.25)
plt.show()

print("Silhouette scores:", silhouette_scores)


## 2. K-Means customer segmentation

Four clusters are used for the final business segmentation because they provide a simple and interpretable four-group structure for this internship project. The silhouette scores are shown above so the clustering quality can be evaluated rather than hiding the model-selection step.


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
df["Cluster"] = kmeans.fit_predict(X)

cluster_profile = df.groupby("Cluster")[features].mean().round(2)
cluster_profile["Customers"] = df["Cluster"].value_counts().sort_index()
display(cluster_profile)


## 3. Create understandable segment names

The names below are descriptive business labels derived from the actual cluster profiles. They are not claims about customer loyalty or purchase history beyond what this dataset can support.


In [ ]:
profile = df.groupby("Cluster")[features].mean()

spend_order = profile["Total Amount"].sort_values(ascending=False).index.tolist()
segment_names = {}

segment_names[spend_order[0]] = "High Spenders"
segment_names[spend_order[1]] = "Mid-Value Shoppers"

remaining = spend_order[2:]
q_order = profile.loc[remaining, "Quantity"].sort_values(ascending=False).index.tolist()
segment_names[q_order[0]] = "Value / High-Quantity Shoppers"
segment_names[q_order[1]] = "Low-Value Shoppers"

df["Segment"] = df["Cluster"].map(segment_names)

segment_summary = df.groupby("Segment").agg(
    Customers=("Customer ID", "count"),
    Avg_Age=("Age", "mean"),
    Avg_Quantity=("Quantity", "mean"),
    Avg_Price=("Price per Unit", "mean"),
    Avg_Total_Amount=("Total Amount", "mean")
).round(2).sort_values("Avg_Total_Amount", ascending=False)

display(segment_summary)


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="Quantity",
    y="Total Amount",
    hue="Segment",
    s=90
)
plt.title("Customer Segments: Quantity vs Total Amount")
plt.xlabel("Quantity Purchased")
plt.ylabel("Total Amount")
plt.grid(alpha=.2)
plt.show()


**Observation:** The scatter plot shows how the segments differ in transaction size and quantity. This is useful for deciding whether a campaign should focus on increasing basket size, promoting higher-value products, or attracting lower-value shoppers.


In [ ]:
plt.figure(figsize=(10, 6))
order = segment_summary.index
sns.barplot(data=df, x="Segment", y="Total Amount", order=order, errorbar=None)
plt.title("Average Transaction Value by Customer Segment")
plt.ylabel("Average Total Amount")
plt.xlabel("")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


**Observation:** The segment averages make the differences in customer value easy to compare and can support differentiated marketing budgets.


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="Segment", y="Age", order=order)
plt.title("Age Distribution Across Customer Segments")
plt.xlabel("")
plt.ylabel("Age")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 4. Actionable business recommendations

### High Spenders
- Promote premium products and higher-margin offers.
- Use personalised recommendations and VIP-style benefits.

### Mid-Value Shoppers
- Encourage larger baskets through bundles and cross-selling.
- Test targeted discounts on complementary products.

### Value / High-Quantity Shoppers
- Offer quantity-based bundles and multi-buy promotions.
- Highlight products where buying more units creates clear value.

### Low-Value Shoppers
- Use low-cost personalised offers to increase transaction value.
- Test entry-level bundles and category recommendations.

**Caution:** These are segment-level recommendations. They should be validated with future transaction data before being used as permanent customer rules.


In [ ]:
# Save the customer-level segmentation for further analysis
df.to_csv("../output/charts/customer_segments.csv", index=False)

# Save the segment summary
segment_summary.to_csv("../output/charts/segment_summary.csv")

print("Customer segmentation completed successfully.")
print("Customers:", len(df))
print("Segments:", df["Segment"].value_counts().to_dict())


## Conclusion

This project converted the retail transaction dataset into four interpretable customer groups using K-Means clustering. The analysis used only variables that are genuinely available in the dataset, avoiding unsupported RFM assumptions.

The resulting segments can help a retailer move from one-size-fits-all marketing toward more targeted offers based on spending, quantity, price level and customer age.
